### TO DO - reunião 22/04/2026

Especificações após reunião realizada com a equipe de DIMAP-1

1. acrescentar informações dos lotes condominiais (apartamento) vinculando eles aos polígonos
2. fazer o melt dos lotes que tem mais de um polígono, transformando em multipoligono
3. criar a base indexada por Setor, Quadra, Lote, Condominio, Tipo de Lote (ou seja, pelo lote em si seja apartamento ou não) - aí os lotes de apartamento herdam o zonemento do polígono do lote do condominio
4. A base final, indexada por lote, deve ter uma coluna para cada zoneamento existente. dentro dessa coluna, um número de 1 a 100 que representa a importância daqueel zoneamento (como já calculamos) para aquele lote. A soma das importancias de todas as zonas deve ser igual a 100
5. A base final deve ser nesse formato:

| Setor | Quadra | Lote | Condomínio | Tipo de Lote | ZOE | ZC | ZEU |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 001 | 015 | 0024 | 08 | F | 20 | 30 | 50 |
| 002 | 042 | 1250 | 15 | M | 10 | 80 | 10 |
| 003 | 088 | 0500 | 22 | V | 45 | 45 | 10 |
| 004 | 102 | 9999 | 40 | F | 60 | 20 | 20 |
| 005 | 067 | 0001 | 99 | M | 33 | 33 | 34 |

6. Acrescentar na base uma coluna de observações (para falar se o lote é multipoligono, etc.) e também colocar a área total do lote e a área total intersectada pelo zoneamento

Enviar também a base indexada por id do polígono para o Bruno Carano.

In [723]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from utils.download_file_geosampa import download_file_geosampa
from utils.io.parquet import load_parquet, save_parquet
import os
from config import FILE_IPTU_GEOSAMPA, DATA_DIR, OUTPUT_DIR

Vou voltar no dataframe original (não o do dashboard) porque vamos ter que recalcular algumas coisas que é melhor fazer depois que der join com os apartamentos.

In [724]:
df = load_parquet('df_final.parquet', subfolder='microdados_lotes_com_zoneamento', gdf=False, output=True)

In [725]:
df.head()

,id_pol_lote,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,area_pol_lote,area_interseccao,id_perimetro_zoneamento,percentual_interseccao,sql
0,5967194,210,008,0053,00,F,ZMa,2024,270.751684,270.751684,536791,100.0,210.008.0053-00
1,5967202,210,011,0047,00,F,ZMa,2024,176.359862,176.359862,536783,100.0,210.011.0047-00
2,5967211,210,016,0099,00,F,ZEUa,2024,150.331192,150.331192,528202,100.0,210.016.0099-00
3,5967212,210,011,0087,00,F,ZMa,2024,145.726729,145.726729,536783,100.0,210.011.0087-00
4,5967247,210,008,0110,00,F,ZMa,2024,161.244863,161.244863,536791,100.0,210.008.0110-00


In [726]:
def load_iptu()->pd.DataFrame:
    
    fname_iptu = 'iptu_2026.parquet'
    fpath_iptu = os.path.join(DATA_DIR, 'iptu', fname_iptu)
    if os.path.exists(fpath_iptu):
        print(f"Arquivo encontrado em {fpath_iptu}. Carregando...")
       
        return load_parquet(fpath_iptu, subfolder='iptu', gdf=False, output=False)
    else:
        print(f"Arquivo não encontrado em {fpath_iptu}. Baixando do GeoSampa...")
        df = download_file_geosampa(FILE_IPTU_GEOSAMPA)
        save_parquet(df, 'iptu_2026.parquet', subfolder='iptu', output=False)
        return df

In [727]:
iptu = load_iptu()

Arquivo encontrado em /home/hpougy/projects/zoneamento_lotes_pmsp/data/iptu/iptu_2026.parquet. Carregando...


In [728]:
iptu.head()

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,ANO DA CONSTRUCAO CORRIGIDO,QUANTIDADE DE PAVIMENTOS,TESTADA PARA CALCULO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE
0,0010030001-4,2026,1,01/01/26,00-0,03812-1,R S CAETANO,13.0,NaN,SANTA EFIGENIA,...,1924,1,13.00,Loja,Comercial horizontal - padrão B,De esquina,0.2,1963,1,0
1,0010030002-2,2026,1,01/01/26,00-0,03812-1,R S CAETANO,19.0,NaN,SANTA EFIGENIA,...,1944,1,6.00,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
2,0010030003-0,2026,1,01/01/26,00-0,03812-1,R S CAETANO,27.0,NaN,SANTA EFIGENIA,...,1965,2,7.85,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
3,0010030004-9,2026,1,01/01/26,00-0,03812-1,R S CAETANO,33.0,NaN,NaN,...,1944,1,6.05,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
4,0010030005-7,2026,1,01/01/26,00-0,03812-1,R S CAETANO,39.0,NaN,NaN,...,2004,2,6.70,Loja,Comercial horizontal - padrão B,Normal,0.8,1963,1,0


In [729]:
iptu.shape

(3920972, 29)

In [730]:
iptu.columns

Index(['NUMERO DO CONTRIBUINTE', 'ANO DO EXERCICIO', 'NUMERO DA NL',
       'DATA DO CADASTRAMENTO', 'NUMERO DO CONDOMINIO', 'CODLOG DO IMOVEL',
       'NOME DE LOGRADOURO DO IMOVEL', 'NUMERO DO IMOVEL',
       'COMPLEMENTO DO IMOVEL', 'BAIRRO DO IMOVEL', 'REFERENCIA DO IMOVEL',
       'CEP DO IMOVEL', 'QUANTIDADE DE ESQUINAS/FRENTES', 'FRACAO IDEAL',
       'AREA DO TERRENO', 'AREA CONSTRUIDA', 'AREA OCUPADA',
       'VALOR DO M2 DO TERRENO', 'VALOR DO M2 DE CONSTRUCAO',
       'ANO DA CONSTRUCAO CORRIGIDO', 'QUANTIDADE DE PAVIMENTOS',
       'TESTADA PARA CALCULO', 'TIPO DE USO DO IMOVEL',
       'TIPO DE PADRAO DA CONSTRUCAO', 'TIPO DE TERRENO',
       'FATOR DE OBSOLESCENCIA', 'ANO DE INICIO DA VIDA DO CONTRIBUINTE',
       'MES DE INICIO DA VIDA DO CONTRIBUINTE', 'FASE DO CONTRIBUINTE'],
      dtype='str')

In [731]:
iptu['endereco'] = iptu['NOME DE LOGRADOURO DO IMOVEL'] + ', ' + iptu['NUMERO DO IMOVEL'].fillna('').astype(str) + ' - '\
      + iptu['BAIRRO DO IMOVEL'].fillna('').astype(str) + ', São Paulo - SP. CEP: ' + iptu['CEP DO IMOVEL'].fillna('') .astype(str)

In [732]:
iptu['endereco'].sample(5)

811846     R  NAPOLEAO DE BARROS, 98.0 - , São Paulo - SP...
1641445    R CONS MOREIRA DE BARROS, 1152.0 - BLOCO B ED ...
1682594    R  ISABEL DE SIQUEIRA BARROS, 501.0 - TORRE B,...
457430     AV S GABRIEL, 626.0 - , São Paulo - SP. CEP: 0...
75403      R CD DE SARZEDAS, 135.0 - , São Paulo - SP. CE...
Name: endereco, dtype: str

In [733]:
iptu['sql'] = iptu['NUMERO DO CONTRIBUINTE'].str[:3] + '.' \
            + iptu['NUMERO DO CONTRIBUINTE'].str[3:6] + '.'  \
            + iptu['NUMERO DO CONTRIBUINTE'].str[6:]

In [734]:
iptu['sql'].sample(5)

61761      005.035.0859-4
3342859    172.128.0055-3
517099     020.091.0586-9
2270630    101.613.0678-1
130386     007.025.0528-5
Name: sql, dtype: str

In [735]:
#lotes condominiais tem numero do condominio igual a "00-0"
lotes_condominiais = iptu[iptu['NUMERO DO CONDOMINIO']!="00-0"]
lotes_condominiais.head()

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,TESTADA PARA CALCULO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE,endereco,sql
45,0010030079-0,2026,1,01/01/26,02-7,18993-6,AV TIRADENTES,262.0,NaN,NaN,...,7.45,Loja em edifício em condomínio (unidade autônoma),Comercial vertical - padrão B,Normal,0.28,1963,1,0,"AV TIRADENTES, 262.0 - , São Paulo - SP. CEP:...",001.003.0079-0
46,0010030080-4,2026,1,01/01/26,02-7,18993-6,AV TIRADENTES,258.0,A1 APTO 1,LUZ,...,7.45,Apartamento em condomínio,Residencial vertical - padrão C,Normal,0.28,1963,1,0,"AV TIRADENTES, 258.0 - LUZ, São Paulo - SP. C...",001.003.0080-4
47,0010030081-2,2026,1,01/01/26,02-7,18993-6,AV TIRADENTES,258.0,A2 APTO 2,LUZ,...,7.45,Apartamento em condomínio,Residencial vertical - padrão C,Normal,0.28,1963,1,0,"AV TIRADENTES, 258.0 - LUZ, São Paulo - SP. C...",001.003.0081-2
48,0010030082-0,2026,1,01/01/26,02-7,18993-6,AV TIRADENTES,258.0,APTO 3,NaN,...,7.45,Apartamento em condomínio,Residencial vertical - padrão C,Normal,0.28,1963,1,0,"AV TIRADENTES, 258.0 - , São Paulo - SP. CEP:...",001.003.0082-0
49,0010030083-9,2026,1,01/01/26,02-7,18993-6,AV TIRADENTES,258.0,APTO 4,LUZ,...,7.45,Apartamento em condomínio,Residencial vertical - padrão C,Normal,0.28,1963,1,0,"AV TIRADENTES, 258.0 - LUZ, São Paulo - SP. C...",001.003.0083-9


O tipo de uso que prevalece é apartamento

In [736]:
lotes_condominiais['TIPO DE USO DO IMOVEL'].value_counts(normalize=True)*100

TIPO DE USO DO IMOVEL
Apartamento em condomínio                                                                                                                             75.709106
Garagem (unidade autônoma) em edifício em condomínio de uso exclusivamente residencial                                                                 9.224104
Escritório/consultório em condomínio (unidade autônoma)                                                                                                6.420893
Residência                                                                                                                                             2.770960
Garagem (unidade autônoma) em edifício em condomínio de escritórios, consultórios ou misto                                                             1.817899
Flat de uso comercial (semelhante a hotel)                                                                                                             1.469041
Loja em edifício e

In [737]:
#vou usar a base principal porque quero puxar as informações de endereço para ela então vou juntar todo mundo mesmo
del lotes_condominiais

Para juntar preciso criar um id

In [738]:
df['id_lote'] = df['cd_setor_fiscal'].astype(str).str.zfill(3) + '.'\
    + df['cd_quadra_fiscal'].astype(str).str.zfill(3) + '.' \
    + df['cd_lote'].astype(str).str.zfill(4) + '-' \
    + df['cd_tipo_lote'].astype(str) + '.' \
    + df['cd_condominio'].astype(str).str.zfill(2)

In [739]:
df['id_lote'].str.len().value_counts()

id_lote
17    1768199
Name: count, dtype: int64

In [740]:
#ainda tem duplicatas porque tem lotes que tem mais de um tipo de uso
df['id_lote'].duplicated().any()

np.True_

Como a base original é resultado da junção de polígonos de lote com polígonos de zoneamento e como vimos no EDA temos lotes com mais de um polígono e também há casos de o mesmo zoneamento também ter mais de um polígono (vide caso da USP nas praças e canteiros ou mesmo na ZOE) preciso fazer o groupby dos dados por id_lote + cd_zoneamento . Ou seja juntar as áreas de intersecção para cada lote e para cada zoneamento que esse lote estiver relacionado.

In [741]:
df['id_lote_zoneamento'] = df['id_lote'] + '.' + df['cd_zoneamento_perimetro']

In [742]:
df['id_lote_zoneamento'].duplicated().sum()

np.int64(9713)

In [743]:
df[df['id_lote_zoneamento'].duplicated(keep=False)]

,id_pol_lote,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,area_pol_lote,area_interseccao,id_perimetro_zoneamento,percentual_interseccao,sql,id_lote,id_lote_zoneamento
379,5981832,210,005,0002,00,M,ZEPAM,2024,6122.583524,3021.670679,524868,49.4,210.005.0002-00,210.005.0002-M.00,210.005.0002-M.00.ZEPAM
380,5981832,210,005,0002,00,M,ZEPAM,2024,6122.583524,3100.912844,559147,50.6,210.005.0002-00,210.005.0002-M.00,210.005.0002-M.00.ZEPAM
488,5985508,210,005,0015,00,F,ZEPAM,2024,23629.297382,29.162476,524868,0.1,210.005.0015-00,210.005.0015-F.00,210.005.0015-F.00.ZEPAM
489,5985508,210,005,0015,00,F,ZEPAM,2024,23629.297382,23600.134906,559147,99.9,210.005.0015-00,210.005.0015-F.00,210.005.0015-F.00.ZEPAM
557,5988069,210,001,0022,00,F,ZMa,2024,452.063729,61.082342,536947,13.5,210.001.0022-00,210.001.0022-F.00,210.001.0022-F.00.ZMa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1767968,7280561,039,043,0000,17,F,ZM,2024,4145.129528,0.007574,551281,0.0,039.043.0000-17,039.043.0000-F.17,039.043.0000-F.17.ZM
1767969,7280561,039,043,0000,17,F,ZM,2024,4145.129528,146.769723,551291,3.5,039.043.0000-17,039.043.0000-F.17,039.043.0000-F.17.ZM
1767970,7280561,039,043,0000,17,F,ZM,2024,4145.129528,0.025896,551292,0.0,039.043.0000-17,039.043.0000-F.17,039.043.0000-F.17.ZM
1768160,5499830,272,002,0011,00,F,ZMa,2024,200.405937,0.001997,536097,0.0,272.002.0011-00,272.002.0011-F.00,272.002.0011-F.00.ZMa


Entao precisamos refazer a area de interseccao da zona considerando os vários perímetros de zoneamento para cada lote e para cada tipo de zoneamento (ou seja, agrupando pelo id_lote_zoneamento)

In [744]:
area_interseccao_zona = df.groupby('id_lote_zoneamento')['area_interseccao'].sum().reset_index()
area_interseccao_zona.head()

,id_lote_zoneamento,area_interseccao
0,001.001.0002-M.00.Praça/Canteiro,1652.533133
1,001.001.0002-M.00.ZC,753.836627
2,001.001.0003-M.00.Praça/Canteiro,4375.522818
3,001.002.0001-M.00.Praça/Canteiro,3843.730094
4,001.002.0002-M.00.Praça/Canteiro,1252.565401


E precisamos também refazer a área do lote, agrupando pelo id do lote (para os casos em que há mais de um polígono de lote)

In [745]:
area_lote = df.groupby('id_lote')['area_pol_lote'].sum().reset_index()
area_lote.head()

,id_lote,area_pol_lote
0,001.001.0002-M.00,4812.882675
1,001.001.0003-M.00,32318.502120
2,001.002.0001-M.00,3848.083386
3,001.002.0002-M.00,1252.664650
4,001.003.0000-F.01,337.361339


Tambem precisamos listar os ids dos perímetros do zoneamento e do lote para poder buscar depois

In [746]:
pols_lote = df.groupby('id_lote')['id_pol_lote'].apply(lambda x: '; '.join(map(str, x))).reset_index()
pols_lote.head()

,id_lote,id_pol_lote
0,001.001.0002-M.00,2054979; 2054979
1,001.001.0003-M.00,2061298; 2061298; 2061298; 2061298; 2061298
2,001.002.0001-M.00,2020230
3,001.002.0002-M.00,2018598
4,001.003.0000-F.01,1985952


In [747]:
pols_zona = df.groupby('id_lote_zoneamento')['id_perimetro_zoneamento'].apply(lambda x: '; '.join(map(str, x))).reset_index()
pols_zona.head()

,id_lote_zoneamento,id_perimetro_zoneamento
0,001.001.0002-M.00.Praça/Canteiro,502193
1,001.001.0002-M.00.ZC,517011
2,001.001.0003-M.00.Praça/Canteiro,500756; 500757; 500758; 500789; 500790
3,001.002.0001-M.00.Praça/Canteiro,502044
4,001.002.0002-M.00.Praça/Canteiro,502045


In [748]:
df.head()

,id_pol_lote,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,area_pol_lote,area_interseccao,id_perimetro_zoneamento,percentual_interseccao,sql,id_lote,id_lote_zoneamento
0,5967194,210,008,0053,00,F,ZMa,2024,270.751684,270.751684,536791,100.0,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa
1,5967202,210,011,0047,00,F,ZMa,2024,176.359862,176.359862,536783,100.0,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa
2,5967211,210,016,0099,00,F,ZEUa,2024,150.331192,150.331192,528202,100.0,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa
3,5967212,210,011,0087,00,F,ZMa,2024,145.726729,145.726729,536783,100.0,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa
4,5967247,210,008,0110,00,F,ZMa,2024,161.244863,161.244863,536791,100.0,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa


In [749]:
df_final = df.drop(columns=['area_interseccao', 'area_pol_lote', 'id_pol_lote', 'id_perimetro_zoneamento', 
                            'percentual_interseccao'])
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa


Tem duplicatas porque tinha aquela questão de ter mias de um poligono de lote por lote assim como mais de um poligono de zoneamento de um mesmo tipo por lote. Entao precisa "jogar fora" elas

In [750]:
df_final.duplicated().sum()

np.int64(9713)

In [751]:
df_final.drop_duplicates(inplace=True)

In [752]:
df_final = df_final.merge(area_interseccao_zona, on='id_lote_zoneamento', how='left')
df_final = df_final.merge(area_lote, on='id_lote', how='left')
df_final = df_final.merge(pols_lote, on='id_lote', how='left')
df_final = df_final.merge(pols_zona, on='id_lote_zoneamento', how='left')

In [753]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa,270.751684,270.751684,5967194,536791
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa,176.359862,176.359862,5967202,536783
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa,150.331192,150.331192,5967211,528202
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa,145.726729,145.726729,5967212,536783
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa,161.244863,161.244863,5967247,536791


In [754]:
df_final.duplicated().any()

np.False_

In [755]:
df_final['percentual_interseccao'] = df_final['area_interseccao'] / df_final['area_pol_lote'] * 100
df_final.head() 

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento,percentual_interseccao
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa,270.751684,270.751684,5967194,536791,100.0
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa,176.359862,176.359862,5967202,536783,100.0
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa,150.331192,150.331192,5967211,528202,100.0
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa,145.726729,145.726729,5967212,536783,100.0
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa,161.244863,161.244863,5967247,536791,100.0


In [756]:
df_final['percentual_interseccao'].max()

np.float64(100.00000000020022)

In [757]:
df_final['percentual_interseccao'].min()

np.float64(2.1111686171569218e-19)

In [758]:
df_final['percentual_interseccao'] = df_final['percentual_interseccao'].round(4)

In [759]:
df_final['percentual_interseccao'].min()

np.float64(0.0)

In [760]:
df_final['interseccao_irrisoria'] = df_final['percentual_interseccao'] == 0

In [761]:
df_final['interseccao_irrisoria'].mean()

np.float64(0.001962483636491846)

In [762]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento,percentual_interseccao,interseccao_irrisoria
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa,270.751684,270.751684,5967194,536791,100.0,False
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa,176.359862,176.359862,5967202,536783,100.0,False
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa,150.331192,150.331192,5967211,528202,100.0,False
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa,145.726729,145.726729,5967212,536783,100.0,False
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa,161.244863,161.244863,5967247,536791,100.0,False


OK mas esse dataframe ainda está indexado apenas para os lotes não-condominaiis. tenho que adicionar as outras informações de lote (tipo endereço) e também acrescentar os apartamentos

In [763]:
#esse é o nosso id
df_final['id_lote'].sample(3)

1588118    064.082.0033-F.00
455085     165.224.0013-F.00
1148199    140.003.0028-F.00
Name: id_lote, dtype: str

In [764]:
iptu.columns

Index(['NUMERO DO CONTRIBUINTE', 'ANO DO EXERCICIO', 'NUMERO DA NL',
       'DATA DO CADASTRAMENTO', 'NUMERO DO CONDOMINIO', 'CODLOG DO IMOVEL',
       'NOME DE LOGRADOURO DO IMOVEL', 'NUMERO DO IMOVEL',
       'COMPLEMENTO DO IMOVEL', 'BAIRRO DO IMOVEL', 'REFERENCIA DO IMOVEL',
       'CEP DO IMOVEL', 'QUANTIDADE DE ESQUINAS/FRENTES', 'FRACAO IDEAL',
       'AREA DO TERRENO', 'AREA CONSTRUIDA', 'AREA OCUPADA',
       'VALOR DO M2 DO TERRENO', 'VALOR DO M2 DE CONSTRUCAO',
       'ANO DA CONSTRUCAO CORRIGIDO', 'QUANTIDADE DE PAVIMENTOS',
       'TESTADA PARA CALCULO', 'TIPO DE USO DO IMOVEL',
       'TIPO DE PADRAO DA CONSTRUCAO', 'TIPO DE TERRENO',
       'FATOR DE OBSOLESCENCIA', 'ANO DE INICIO DA VIDA DO CONTRIBUINTE',
       'MES DE INICIO DA VIDA DO CONTRIBUINTE', 'FASE DO CONTRIBUINTE',
       'endereco', 'sql'],
      dtype='str')

In [765]:
iptu['sql']

0          001.003.0001-4
1          001.003.0002-2
2          001.003.0003-0
3          001.003.0004-9
4          001.003.0005-7
                ...      
3920967    310.118.0238-1
3920968    310.118.0239-1
3920969    310.118.0240-3
3920970    310.119.0001-4
3920971    310.120.0001-7
Name: sql, Length: 3920972, dtype: str

In [766]:
iptu['id_join'] = iptu['sql'].str.split('-').str[0]
iptu['id_join'].sample(5)

822947     038.009.0699
3409751    179.007.0046
1521641    068.172.0010
1570906    070.078.0102
3249595    169.027.0017
Name: id_join, dtype: object

In [767]:
iptu['NUMERO DO CONDOMINIO'].sample()

1621715    03-5
Name: NUMERO DO CONDOMINIO, dtype: str

In [768]:
df_final['id_lote'].sample(3)

449872    165.099.0208-F.00
209978    248.211.0021-F.00
532240    172.142.0043-F.00
Name: id_lote, dtype: str

In [769]:
iptu['id_join'] = iptu['id_join'] + '-F.' + iptu['NUMERO DO CONDOMINIO'].str.split('-').str[0]
iptu['id_join'].sample(5)

3787425    299.046.0911-F.09
635368     028.028.0112-F.00
1277918    055.399.0055-F.00
2543335    116.541.0109-F.00
3809225    299.110.0377-F.06
Name: id_join, dtype: object

Vemos que todos os lotes condominiais da base de polígonos tem o lote marcado como '0000'

In [770]:
condominiais_base_poligonos = df_final[~df_final['id_lote'].str.endswith('00')]['id_lote']
condominiais_base_poligonos.sample(5)

1417685    009.026.0000-F.03
1137555    031.088.0000-F.01
1062925    081.207.0000-F.04
1064553    096.026.0000-F.01
393111     118.196.0000-F.01
Name: id_lote, dtype: str

In [771]:
(condominiais_base_poligonos.str[8:12]=='0000').all()

np.True_

Entao temos que fazer o mesmo para os lotes condominiais da base do IPTU, trocando a informação do lote por 00

In [772]:
# o lote está no id 8:12
lote_teste = '261.146.0088-F.01'
lote_teste[8:12]

'0088'

In [773]:
#fazendo isso eu troco ele para 0000
lote_teste[:8] + '0000' + lote_teste[12:]

'261.146.0000-F.01'

Nesses casos, preciso substituir a informação do "lote" por 0000 para bater com o que está na base de polígonos (que é o que puxei do geosampa)

In [774]:
def ajustar_id_lotes_fiscais(lote_id:str):

    if not lote_id.endswith('00'):
        lote_id_ajustado = lote_id[:8] + '0000' + lote_id[12:]
        return lote_id_ajustado
    else:
        return lote_id

In [775]:
iptu['id_join'] = iptu['id_join'].apply(ajustar_id_lotes_fiscais)

Ainda assim há alguns lotes da base fiscal do IPTU que não conseguiram ser encontrados na base de polígonos.

In [776]:
lotes_iptu_nao_encontrados = (set(iptu['id_join'].unique())-set(df_final['id_lote'].unique()))


In [777]:
all([lote[-2:]=='00' for lote in lotes_iptu_nao_encontrados])

False

Um percentual muito pequeno são condominios, a grande maioria nao são. Então sabemos que o problema não é a lógica que usamos para substituir o lote por 0000.

In [778]:
nao_encontrados_condominio = [lote[-2:]!='00' for lote in lotes_iptu_nao_encontrados]
nao_encontrados_nao_condominio = [lote[-2:]=='00' for lote in lotes_iptu_nao_encontrados]

sum(nao_encontrados_condominio)/len(lotes_iptu_nao_encontrados)

0.022180436701331595

In [779]:
sum(nao_encontrados_condominio)

578

In [780]:
sum(nao_encontrados_nao_condominio)/len(lotes_iptu_nao_encontrados)

0.9778195632986684

In [781]:
base_iptu_nao_encontrados = iptu[iptu['id_join'].isin(lotes_iptu_nao_encontrados)]
base_iptu_nao_encontrados.sample(5)

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE,endereco,sql,id_join
2528401,1162870053-9,2026,1,01/01/26,00-0,05685-5,R DAMIAO DE GOIS,63.0,NaN,NaN,...,Residência,Residencial horizontal - padrão C,Normal,0.44,2006,1,0,"R DAMIAO DE GOIS, 63.0 - , São Paulo - SP. CE...",116.287.0053-9,116.287.0053-F.00
1451151,0661190091-8,2026,1,01/01/26,01-9,06428-9,R IRMA EMERENCIANA,800.0,AP 46,TUCURUVI,...,Apartamento em condomínio,Residencial vertical - padrão C,Normal,0.97,2022,9,0,"R IRMA EMERENCIANA, 800.0 - TUCURUVI, São Paul...",066.119.0091-8,066.119.0000-F.01
1481999,0671450030-0,2026,1,01/01/26,00-0,13745-6,R SILVESTRE LACROIX,152.0,NaN,VL CONSTANCA,...,Residência,Residencial horizontal - padrão C,Normal,0.76,2001,1,0,"R SILVESTRE LACROIX, 152.0 - VL CONSTANCA, Sã...",067.145.0030-0,067.145.0030-F.00
3133123,1603700515-9,2026,1,01/01/26,04-3,46916-5,R DR HELIO FIDELIS,26.0,AP 51,TORRE B,...,Apartamento em condomínio,Residencial vertical - padrão C,De duas ou mais frentes,0.85,2010,1,0,"R DR HELIO FIDELIS, 26.0 - TORRE B, São Paulo ...",160.370.0515-9,160.370.0000-F.04
3621793,2150420271-4,2026,1,01/01/26,02-7,06361-4,AV ELISIO TEIXEIRA LEITE,6300.0,AP 62,BLOCO D,...,Apartamento em condomínio,Residencial vertical - padrão B,De duas ou mais frentes,0.66,2023,2,0,"AV ELISIO TEIXEIRA LEITE, 6300.0 - BLOCO D, S...",215.042.0271-4,215.042.0000-F.02


In [782]:
save_parquet(base_iptu_nao_encontrados, 'base_iptu_nao_encontrados.parquet', subfolder='nao_encontrados', output=True, gdf=False)

PosixPath('/home/hpougy/projects/zoneamento_lotes_pmsp/output/nao_encontrados/base_iptu_nao_encontrados.parquet')

In [783]:
poligonos_nao_encontrados_iptu = set(df_final['id_lote'].unique()) - set(iptu['id_join'].unique())
len(poligonos_nao_encontrados_iptu)

34953

Também há 2% de polígonos que não possuem vínculo com os dados do IPTU. Mas nesse caso é até que esperado pois os dados do IPTU são apenas lotes F - Fiscais

In [784]:
len(poligonos_nao_encontrados_iptu)/len(df_final['id_lote'].unique())

0.020757663938807264

No entanto 40% dos nao encontrados sao lotes fiscais

In [785]:
df_final[df_final['id_lote'].isin(poligonos_nao_encontrados_iptu)]['cd_tipo_lote'].value_counts(normalize=True)


cd_tipo_lote
M    0.505085
F    0.401577
V    0.093338
Name: proportion, dtype: float64

In [786]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento,percentual_interseccao,interseccao_irrisoria
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa,270.751684,270.751684,5967194,536791,100.0,False
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa,176.359862,176.359862,5967202,536783,100.0,False
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa,150.331192,150.331192,5967211,528202,100.0,False
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa,145.726729,145.726729,5967212,536783,100.0,False
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa,161.244863,161.244863,5967247,536791,100.0,False


In [787]:
df_final['poligono_fiscal_sem_lote_iptu'] = (df_final['id_lote'].isin(poligonos_nao_encontrados_iptu))&(df_final['cd_tipo_lote']=='F')

De todo modo esses polígonos de lotes fiscais são muito pouco considerando o total dos polígonos... menos de 1%

In [788]:
df_final['poligono_fiscal_sem_lote_iptu'].mean()

np.float64(0.00909418670378951)

In [789]:
poligonos_nao_encontrados_iptu = df_final[df_final['poligono_fiscal_sem_lote_iptu']]
poligonos_nao_encontrados_iptu.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento,percentual_interseccao,interseccao_irrisoria,poligono_fiscal_sem_lote_iptu
28,210,016,0031,00,F,ZEUa,2024,210.016.0031-00,210.016.0031-F.00,210.016.0031-F.00.ZEUa,268.960690,268.960690,5968281,528204,100.0,False,True
37,210,016,0044,00,F,ZEUa,2024,210.016.0044-00,210.016.0044-F.00,210.016.0044-F.00.ZEUa,501.109732,501.109732,5968418,528202,100.0,False,True
57,210,008,0040,00,F,ZMa,2024,210.008.0040-00,210.008.0040-F.00,210.008.0040-F.00.ZMa,173.979538,173.979538,5969283,536791,100.0,False,True
110,210,023,0030,00,F,ZEUa,2024,210.023.0030-00,210.023.0030-F.00,210.023.0030-F.00.ZEUa,456.117128,456.117128,5971600,528200,100.0,False,True
141,210,015,0009,00,F,ZMa,2024,210.015.0009-00,210.015.0009-F.00,210.015.0009-F.00.ZMa,258.657264,258.657264,5972537,536948,100.0,False,True


In [790]:
len(poligonos_nao_encontrados_iptu)

15992

In [791]:
save_parquet(poligonos_nao_encontrados_iptu, 'registros_fiscais_nao_encontados_iptu.parquet', subfolder='nao_encontrados', output=True, gdf=False)

PosixPath('/home/hpougy/projects/zoneamento_lotes_pmsp/output/nao_encontrados/registros_fiscais_nao_encontados_iptu.parquet')

Ok agora vamos dar o merge

In [792]:
iptu.columns

Index(['NUMERO DO CONTRIBUINTE', 'ANO DO EXERCICIO', 'NUMERO DA NL',
       'DATA DO CADASTRAMENTO', 'NUMERO DO CONDOMINIO', 'CODLOG DO IMOVEL',
       'NOME DE LOGRADOURO DO IMOVEL', 'NUMERO DO IMOVEL',
       'COMPLEMENTO DO IMOVEL', 'BAIRRO DO IMOVEL', 'REFERENCIA DO IMOVEL',
       'CEP DO IMOVEL', 'QUANTIDADE DE ESQUINAS/FRENTES', 'FRACAO IDEAL',
       'AREA DO TERRENO', 'AREA CONSTRUIDA', 'AREA OCUPADA',
       'VALOR DO M2 DO TERRENO', 'VALOR DO M2 DE CONSTRUCAO',
       'ANO DA CONSTRUCAO CORRIGIDO', 'QUANTIDADE DE PAVIMENTOS',
       'TESTADA PARA CALCULO', 'TIPO DE USO DO IMOVEL',
       'TIPO DE PADRAO DA CONSTRUCAO', 'TIPO DE TERRENO',
       'FATOR DE OBSOLESCENCIA', 'ANO DE INICIO DA VIDA DO CONTRIBUINTE',
       'MES DE INICIO DA VIDA DO CONTRIBUINTE', 'FASE DO CONTRIBUINTE',
       'endereco', 'sql', 'id_join'],
      dtype='str')

In [793]:
iptu.rename(columns={'sql':'contribuinte_iptu',
                     'id_join' : 'id_lote'}, inplace=True)

In [794]:
colunas_interesse_iptu = [
    'id_lote',
    'contribuinte_iptu',
    'endereco'
]

In [795]:
iptu = iptu[colunas_interesse_iptu]

In [796]:
df_final = pd.merge(df_final, iptu, on='id_lote', how='left')

No final apenas 0.3% dos polígonos de lotes fiscais ficaram sem contribuinte do IPTU

In [797]:
df_final[(df_final['cd_tipo_lote']=='F')&df_final['contribuinte_iptu'].isnull()].shape[0]/df_final.shape[0]

0.003684864795739087

In [798]:
save_parquet(df_final, 'df_final_com_iptu.parquet', subfolder='microdados_lotes_com_zoneamento', output=True)

PosixPath('/home/hpougy/projects/zoneamento_lotes_pmsp/output/microdados_lotes_com_zoneamento/df_final_com_iptu.parquet')

Agora vou fazer aquela transformação que a Mariana pediu de ter um tipo de zoneamento em cada coluna

In [799]:
df_final.shape

(4339915, 19)

In [800]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,sql,id_lote,id_lote_zoneamento,area_interseccao,area_pol_lote,id_pol_lote,id_perimetro_zoneamento,percentual_interseccao,interseccao_irrisoria,poligono_fiscal_sem_lote_iptu,contribuinte_iptu,endereco
0,210,008,0053,00,F,ZMa,2024,210.008.0053-00,210.008.0053-F.00,210.008.0053-F.00.ZMa,270.751684,270.751684,5967194,536791,100.0,False,False,210.008.0053-7,"R LUIS SALADIN, 187.0 - CID D'ABRIL, São Paul..."
1,210,011,0047,00,F,ZMa,2024,210.011.0047-00,210.011.0047-F.00,210.011.0047-F.00.ZMa,176.359862,176.359862,5967202,536783,100.0,False,False,210.011.0047-4,"R MANUEL BENICIO, 257.0 - , São Paulo - SP. C..."
2,210,016,0099,00,F,ZEUa,2024,210.016.0099-00,210.016.0099-F.00,210.016.0099-F.00.ZEUa,150.331192,150.331192,5967211,528202,100.0,False,False,210.016.0099-1,"R PERA-MARMELO, 472.0 - CID D ABRIL, São Paul..."
3,210,011,0087,00,F,ZMa,2024,210.011.0087-00,210.011.0087-F.00,210.011.0087-F.00.ZMa,145.726729,145.726729,5967212,536783,100.0,False,False,210.011.0087-3,"R MICAELA ALEGRIA, 70.0 - CID D ABRIL, São Pa..."
4,210,008,0110,00,F,ZMa,2024,210.008.0110-00,210.008.0110-F.00,210.008.0110-F.00.ZMa,161.244863,161.244863,5967247,536791,100.0,False,False,210.008.0110-1,"R SIMAO MAYER, 117.0 - CID D ABRIL, São Paulo..."


In [801]:
df_final['percentual_interseccao'].max()

np.float64(100.0)

In [802]:
colunas_interesse = ['cd_setor_fiscal',
           'cd_quadra_fiscal',
           'cd_lote',
           'cd_condominio',
           'cd_tipo_lote',
           'contribuinte_iptu',
           'endereco',
           'id_pol_lote',
           'id_perimetro_zoneamento',
           'cd_zoneamento_perimetro',
           'area_pol_lote',
           'percentual_interseccao']

In [803]:
df_final= df_final[colunas_interesse]
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,contribuinte_iptu,endereco,id_pol_lote,id_perimetro_zoneamento,cd_zoneamento_perimetro,area_pol_lote,percentual_interseccao
0,210,008,0053,00,F,210.008.0053-7,"R LUIS SALADIN, 187.0 - CID D'ABRIL, São Paul...",5967194,536791,ZMa,270.751684,100.0
1,210,011,0047,00,F,210.011.0047-4,"R MANUEL BENICIO, 257.0 - , São Paulo - SP. C...",5967202,536783,ZMa,176.359862,100.0
2,210,016,0099,00,F,210.016.0099-1,"R PERA-MARMELO, 472.0 - CID D ABRIL, São Paul...",5967211,528202,ZEUa,150.331192,100.0
3,210,011,0087,00,F,210.011.0087-3,"R MICAELA ALEGRIA, 70.0 - CID D ABRIL, São Pa...",5967212,536783,ZMa,145.726729,100.0
4,210,008,0110,00,F,210.008.0110-1,"R SIMAO MAYER, 117.0 - CID D ABRIL, São Paulo...",5967247,536791,ZMa,161.244863,100.0


In [804]:
df_final['id'] = df_final.reset_index().index

In [805]:
colunas_dados = [
    'id',
    'cd_zoneamento_perimetro',
    'percentual_interseccao'
]

In [806]:
dados = df_final[colunas_dados].pivot(index='id',columns='cd_zoneamento_perimetro', values='percentual_interseccao').fillna(0)
dados = dados.reset_index()
dados.head()

cd_zoneamento_perimetro,id,AC-1,AC-2,Praça/Canteiro,ZC,ZC-ZEIS,ZCOR-1,ZCOR-2,ZCOR-3,ZCORa,...,ZM,ZMIS,ZMISa,ZMa,ZOE,ZPDS,ZPDSr,ZPI-1,ZPI-2,ZPR
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0


Vou testar a consistencia agora olhando o min e o max por linha - nao pode passar de 100 nem ser menor de 1

In [807]:
total = dados.drop(columns='id').sum(axis=1)

In [808]:
#lembramos do EDA qeu o total pode ser menor porqeu tem lotes que tem viário dentro e o zoneamento nao compreende o viario
total.mean()

np.float64(84.30868439815066)

In [809]:
total.max()

np.float64(100.0)

In [810]:
total.min()

np.float64(0.0)

In [811]:
df_final = df_final.drop(columns=['cd_zoneamento_perimetro', 'percentual_interseccao'])

In [812]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,contribuinte_iptu,endereco,id_pol_lote,id_perimetro_zoneamento,area_pol_lote,id
0,210,008,0053,00,F,210.008.0053-7,"R LUIS SALADIN, 187.0 - CID D'ABRIL, São Paul...",5967194,536791,270.751684,0
1,210,011,0047,00,F,210.011.0047-4,"R MANUEL BENICIO, 257.0 - , São Paulo - SP. C...",5967202,536783,176.359862,1
2,210,016,0099,00,F,210.016.0099-1,"R PERA-MARMELO, 472.0 - CID D ABRIL, São Paul...",5967211,528202,150.331192,2
3,210,011,0087,00,F,210.011.0087-3,"R MICAELA ALEGRIA, 70.0 - CID D ABRIL, São Pa...",5967212,536783,145.726729,3
4,210,008,0110,00,F,210.008.0110-1,"R SIMAO MAYER, 117.0 - CID D ABRIL, São Paulo...",5967247,536791,161.244863,4


In [814]:
df_final = pd.merge(df_final, dados, on='id', how='left')

In [815]:
df_final.head()

,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,contribuinte_iptu,endereco,id_pol_lote,id_perimetro_zoneamento,area_pol_lote,...,ZM,ZMIS,ZMISa,ZMa,ZOE,ZPDS,ZPDSr,ZPI-1,ZPI-2,ZPR
0,210,008,0053,00,F,210.008.0053-7,"R LUIS SALADIN, 187.0 - CID D'ABRIL, São Paul...",5967194,536791,270.751684,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
1,210,011,0047,00,F,210.011.0047-4,"R MANUEL BENICIO, 257.0 - , São Paulo - SP. C...",5967202,536783,176.359862,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
2,210,016,0099,00,F,210.016.0099-1,"R PERA-MARMELO, 472.0 - CID D ABRIL, São Paul...",5967211,528202,150.331192,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,210,011,0087,00,F,210.011.0087-3,"R MICAELA ALEGRIA, 70.0 - CID D ABRIL, São Pa...",5967212,536783,145.726729,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
4,210,008,0110,00,F,210.008.0110-1,"R SIMAO MAYER, 117.0 - CID D ABRIL, São Paulo...",5967247,536791,161.244863,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0


In [817]:
path = save_parquet(df_final, 'df_final_completo.parquet', subfolder='resultado_final', output=True)

Vou salvar em .csv no data (que nao vai pro repo) par poder mandar por email também

In [820]:
from utils.io.path import data_path

In [822]:
path = data_path('df_final.csv', subfolder='resultado_final')

In [823]:
df_final.to_csv(path, sep=';', index=False, encoding='utf-8')

In [824]:
pd.read_csv(path, sep=';').head()

/tmp/ipykernel_41599/926521.py:1: DtypeWarning: Columns (0: id_perimetro_zoneamento) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(path, sep=';').head()


,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,contribuinte_iptu,endereco,id_pol_lote,id_perimetro_zoneamento,area_pol_lote,...,ZM,ZMIS,ZMISa,ZMa,ZOE,ZPDS,ZPDSr,ZPI-1,ZPI-2,ZPR
0,210,8,53,0,F,210.008.0053-7,"R LUIS SALADIN, 187.0 - CID D'ABRIL, São Paul...",5967194,536791,270.751684,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
1,210,11,47,0,F,210.011.0047-4,"R MANUEL BENICIO, 257.0 - , São Paulo - SP. C...",5967202,536783,176.359862,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
2,210,16,99,0,F,210.016.0099-1,"R PERA-MARMELO, 472.0 - CID D ABRIL, São Paul...",5967211,528202,150.331192,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,210,11,87,0,F,210.011.0087-3,"R MICAELA ALEGRIA, 70.0 - CID D ABRIL, São Pa...",5967212,536783,145.726729,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
4,210,8,110,0,F,210.008.0110-1,"R SIMAO MAYER, 117.0 - CID D ABRIL, São Paulo...",5967247,536791,161.244863,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
